# 1. Cleaning and Imputation

In [3]:
import pandas as pd
import numpy as np
import re

## 1. Loading Data and initialization of *Silver Layer*

In [4]:
#path to csv file
file_path = r'C:\Users\Maciek\Desktop\netflixdb\databases\movies.csv'

#Loading raw data
df_bronze = pd.read_csv(file_path, sep = ',')

#Verification
print(f"Number of columns in df: {df_bronze.shape[1]}")
print("First 5 rows: ")
print(df_bronze.head())

#Coping bronze layer into silver layer (df_silver) for secure data transformations
df_silver = df_bronze.copy()



Number of columns in df: 18
First 5 rows: 
     movie_id            title     content_type genre_primary genre_secondary  \
0  movie_0001    Dragon Legend  Stand-up Comedy       History        Thriller   
1  movie_0002    Storm Warrior  Stand-up Comedy        Sci-Fi             NaN   
2  movie_0003      Fire Family            Movie         Drama             NaN   
3  movie_0004     Our Princess      Documentary        Sci-Fi             NaN   
4  movie_0005  Warrior Mission      Documentary         Sport         Mystery   

   release_year  duration_minutes rating  language country_of_origin  \
0          2014              35.0   TV-Y    French             Japan   
1          2017              37.0     PG  Japanese               USA   
2          2003             142.0  TV-MA   English               USA   
3          2011             131.0  NC-17  Japanese               USA   
4          2015              91.0   TV-G   English               USA   

   imdb_rating  production_budget  bo

## 2. Analyses and Exploration Nulls

In [5]:
# Checking number of Nulls (NaN) in each column, to plan imputation
print(df_bronze.isnull().sum())
print(df_bronze.describe())

movie_id                 0
title                    0
content_type             0
genre_primary            0
genre_secondary        667
release_year             0
duration_minutes         0
rating                   0
language                 0
country_of_origin        0
imdb_rating            150
production_budget      675
box_office_revenue     709
number_of_seasons      751
number_of_episodes     719
is_netflix_original      0
added_to_platform        0
content_warning          0
dtype: int64
       release_year  duration_minutes  imdb_rating  production_budget  \
count   1040.000000       1040.000000   890.000000       3.650000e+02   
mean    2006.416346         89.112500     6.268539       1.111541e+07   
std       11.414524         69.298936     1.809088       2.395707e+07   
min     1953.000000          0.000000     0.500000       6.837300e+04   
25%     1998.000000         51.000000     5.300000       1.445488e+06   
50%     2006.000000         81.000000     6.400000       3.7784

In [6]:
#Showing groups in 'genre_secondary'
print(df_bronze.groupby('genre_secondary')['movie_id'].count())

genre_secondary
Action         11
Adventure      25
Animation      17
Biography      20
Comedy         14
Crime          20
Documentary    14
Drama          28
Family         22
Fantasy        20
History        19
Horror         12
Music          14
Mystery        20
Romance        17
Sci-Fi         23
Sport          15
Thriller       23
War            19
Western        20
Name: movie_id, dtype: int64


## 3. Cleaning and Imputation of Categorical Data

In [7]:
# 'genre_secondary' manipulations

#Standarization: Converting empty strings and white signs to NaN
df_silver['genre_secondary'] = df_silver['genre_secondary'].str.strip().replace('',np.nan)

#Handling Non-standard strings 'NULL' as NaN
df_silver['genre_secondary'] = df_silver['genre_secondary'].replace('NULL', np.nan)

#Imputaion: Filling NaN with 'unknown' value. Making business lvl understanding of data
df_silver['genre_secondary'] = df_silver['genre_secondary'].fillna('unknown')

#Verification check
print(df_silver.isnull().sum())

movie_id                 0
title                    0
content_type             0
genre_primary            0
genre_secondary          0
release_year             0
duration_minutes         0
rating                   0
language                 0
country_of_origin        0
imdb_rating            150
production_budget      675
box_office_revenue     709
number_of_seasons      751
number_of_episodes     719
is_netflix_original      0
added_to_platform        0
content_warning          0
dtype: int64


In [8]:
# handling blank spaces in 'title','rating','language','country_of_origin' columns

columns_to_strip = ['title', 'rating', 'language', 'country_of_origin']
for column in columns_to_strip:
    df_silver[column] = df_silver[column].str.strip()

## 4. IMDb Rating Imputation

In [9]:
# Calculate the median from available data. Median is robust to outliers.
imdb_median = df_silver['imdb_rating'].median()

# Feature Engineering: Create a binary flag (1/0) indicating where the original value was missing.
# This is crucial for ML models to capture the predictive power of missing data itself.
df_silver['is_imdb_rating_missing'] = df_silver['imdb_rating'].isna().astype(int)

# Imputation: Fill NaN with the calculated median to preserve the column's statistical distribution.
df_silver['imdb_rating'] = df_silver['imdb_rating'].fillna(imdb_median)
    
#Verification
print(df_silver.isnull().sum())

movie_id                    0
title                       0
content_type                0
genre_primary               0
genre_secondary             0
release_year                0
duration_minutes            0
rating                      0
language                    0
country_of_origin           0
imdb_rating                 0
production_budget         675
box_office_revenue        709
number_of_seasons         751
number_of_episodes        719
is_netflix_original         0
added_to_platform           0
content_warning             0
is_imdb_rating_missing      0
dtype: int64


## 5. Financial Imputation

In [10]:
#Pattern to extract monetary values (e.g., $1,234,567)
pattern_to_clean = r'[$,\s]'

#Copy of the original column for testing purposes
df_test = df_silver['production_budget'].astype(str).copy()

#Replacing characters matching the pattern with an empty string
df_cleaned_data = df_test.str.replace(pattern_to_clean, '', regex = True)

rows_to_clean = df_test.str.len() != df_cleaned_data.str.len()

print(f"Number of rows to clean in 'production_budget': {rows_to_clean.sum()}")

Number of rows to clean in 'production_budget': 0


In [11]:
#Pattern to extract monetary values (e.g., $1,234,567)
pattern_to_clean = r'[$,\s]'

#Copy of the original column for testing purposes
df_test = df_silver['box_office_revenue'].astype(str).copy()

#Replacing characters matching the pattern with an empty string
df_cleaned_data = df_test.str.replace(pattern_to_clean, '', regex = True)

rows_to_clean = df_test.str.len() != df_cleaned_data.str.len()

print(f"Number of rows to clean in 'box_office_revenue': {rows_to_clean.sum()}")

Number of rows to clean in 'box_office_revenue': 0


In [18]:
#Analysing column schema
print(df_silver['production_budget'])

# Flagging: Create the missingness indicator flag (1/0) before imputation.
df_silver['is_production_budget_missing'] = df_silver['production_budget'].isna().astype(int)

# Imputation: Fill NaN with zero (0). For financial data, missing usually means zero or undisclosed, not the average budget.
df_silver['production_budget'] = df_silver['production_budget'].fillna(0)

# Converting 'production_budget' to float with 4 decimal places
df_silver['production_budget'] = df_silver['production_budget'].round(4).astype(float)

0              0.0
1              0.0
2        2114120.0
3              0.0
4              0.0
           ...    
1035           0.0
1036           0.0
1037           0.0
1038     3593765.0
1039    14939487.0
Name: production_budget, Length: 1040, dtype: float64


In [19]:
#Creating binary column to indentify records w/o values
df_silver['is_box_office_revenue_missing'] = df_silver['box_office_revenue'].isna().astype(int)

#Replacing NaN with 0
df_silver['box_office_revenue'] = df_silver['box_office_revenue'].fillna(0)

# Converting 'box_office_revenue' to float with 4 decimal places
df_silver['box_office_revenue'] = df_silver['box_office_revenue'].round(4).astype(float)

#Verification
print(df_silver['box_office_revenue'].isnull().sum())

0


## 6. Series Imputation

In [15]:
series_col = ['number_of_episodes','number_of_seasons']

for col in series_col:
    # Imputation: Fill NaN with 1. This is semantically correct for films/stand-up specials.
    df_silver[col] = df_silver[col].fillna(1)

    # Type Conversion: Ensure the column is an integer type.
    df_silver[col] = df_silver[col].astype(int)

print("\nLiczba NaN po imputacji:")
print(df_silver[series_col].isna().sum()) 
print("\nTypy kolumn po konwersji:")
print(df_silver[series_col].dtypes)


Liczba NaN po imputacji:
number_of_episodes    0
number_of_seasons     0
dtype: int64

Typy kolumn po konwersji:
number_of_episodes    int64
number_of_seasons     int64
dtype: object


# 2. Convertion and Formating

In [16]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   movie_id                       1040 non-null   object 
 1   title                          1040 non-null   object 
 2   content_type                   1040 non-null   object 
 3   genre_primary                  1040 non-null   object 
 4   genre_secondary                1040 non-null   object 
 5   release_year                   1040 non-null   int64  
 6   duration_minutes               1040 non-null   float64
 7   rating                         1040 non-null   object 
 8   language                       1040 non-null   object 
 9   country_of_origin              1040 non-null   object 
 10  imdb_rating                    1040 non-null   float64
 11  production_budget              1040 non-null   float64
 12  box_office_revenue             1040 non-null   f

## 7. Final Output: Saving Silver Layer to CSV

In [12]:
print(df_silver.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   movie_id                       1040 non-null   object 
 1   title                          1040 non-null   object 
 2   content_type                   1040 non-null   object 
 3   genre_primary                  1040 non-null   object 
 4   genre_secondary                1040 non-null   object 
 5   release_year                   1040 non-null   int64  
 6   duration_minutes               1040 non-null   float64
 7   rating                         1040 non-null   object 
 8   language                       1040 non-null   object 
 9   country_of_origin              1040 non-null   object 
 10  imdb_rating                    1040 non-null   float64
 11  production_budget              1040 non-null   float64
 12  box_office_revenue             1040 non-null   f

In [ ]:
# Converting 'added_to_platform' to datetime format
df_silver['added_to_platform'] = pd.to_datetime(df_silver['added_to_platform'], format='%Y-%m-%d')

# 3. Loading data into csv

In [20]:
output_file = 'netflix_silver_layer_movies.csv'

# Save the cleaned DataFrame. index=False is crucial for TSQL BULK INSERT.
df_silver.to_csv(
    output_file,
    sep = ',',
    index=False # Ensure this is set to False for the final output!
)